# EPPO Fine-Tuning: BC Warm Start → PPO with Session Architecture

## Architecture
```
GLOBAL concept pool (all uploaded courses, ~115 concepts)
    │
    │  Student: 'I want to study OS scheduling'
    ▼
SESSION PRE-SELECTION
  cap = MAX_STEPS // 3 = 20
  rank by P(correct|Hard) ascending → take weakest N
  tie-break: LLM difficulty → random
    │
    ▼
SESSION SCOPE (13-20 concepts)
  Agent assigns ONLY within scope
  PFA propagation uses GLOBAL similarity graph
    │
    ▼
GOAL: master max(1, round(not_yet_mastered × 0.30)) NEW concepts
      mastered = P(correct|Hard) > 0.68
```

## Training: 5000 episodes × 60 steps = 288k transitions, ~45 min T4

## Cell 1: Imports

In [30]:
import subprocess
subprocess.run(['pip','install','sentence-transformers','-q'],check=True)

import numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.distributions import Categorical
from sentence_transformers import SentenceTransformer
from collections import defaultdict
import math, warnings, os
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.figsize']=(18,5)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## Cell 2: Global Concept Pool

5 simulated courses (~115 concepts). **In production:** replace COURSES with LLM extractor output from uploaded PDFs.

In [31]:
# (concept_name, llm_difficulty_1_to_5)
COURSES = {
    'algorithms': [
        ('time complexity analysis',2),('space complexity analysis',2),
        ('recursion',2),('divide and conquer',3),('sorting algorithms',2),
        ('merge sort',3),('quick sort',3),('binary search',2),
        ('binary search tree',3),('balanced BST',4),
        ('graph traversal BFS DFS',3),('shortest path Dijkstra',4),
        ('dynamic programming basics',4),('memoization',3),
        ('greedy algorithms',3),('hash table',2),
        ('heap and priority queue',3),('dynamic programming advanced',5),
    ],
    'operating_systems': [
        ('process vs thread',2),('process scheduling',3),
        ('FCFS scheduling',2),('round robin scheduling',3),
        ('priority scheduling',3),('multilevel queue scheduling',4),
        ('process synchronization',4),('mutex and semaphore',3),
        ('deadlock conditions',3),('deadlock prevention',4),
        ('deadlock detection recovery',4),('memory management basics',3),
        ('paging',3),('segmentation',4),('virtual memory',4),
        ('page replacement algorithms',4),('thrashing',5),
        ('file system structure',3),('disk scheduling',3),('I/O management',4),
    ],
    'computer_networks': [
        ('OSI model layers',2),('TCP IP stack',3),
        ('IP addressing',2),('subnetting CIDR',3),
        ('routing fundamentals',3),('routing protocols RIP OSPF',4),
        ('TCP connection management',3),('TCP congestion control',4),
        ('UDP vs TCP',2),('DNS',2),('HTTP HTTPS',2),
        ('network security basics',3),('TLS SSL handshake',4),
        ('firewalls and NAT',3),('wireless networking',3),
        ('network performance metrics',3),('software defined networking',5),
    ],
    'computer_architecture': [
        ('binary and number systems',1),('logic gates',2),
        ('combinational circuits',3),('sequential circuits',3),
        ('CPU datapath',3),('instruction set architecture',3),
        ('CPU pipeline stages',4),('pipeline hazards',4),
        ('branch prediction',4),('cache memory',3),('cache coherence',5),
        ('memory hierarchy',3),('virtual memory hardware',4),
        ('multicore architecture',4),('GPU architecture basics',4),('SIMD parallelism',4),
    ],
    'software_engineering': [
        ('software development lifecycle',2),('agile and scrum',2),
        ('design patterns',3),('SOLID principles',3),
        ('object oriented design',3),('UML diagrams',3),
        ('version control git',2),('testing strategies',3),
        ('unit testing',3),('integration testing',3),
        ('CI CD pipelines',4),('microservices architecture',4),
        ('REST API design',3),('database design',3),
        ('system design scalability',5),('security in software',4),
        ('code review practices',2),('refactoring',3),
    ],
}

GLOBAL_CONCEPTS=[]; GLOBAL_LLM_DIFF=[]; COURSE_CONCEPT_INDICES={}
for cname,concepts in COURSES.items():
    idxs=[]
    for name,diff in concepts:
        GLOBAL_CONCEPTS.append(name); GLOBAL_LLM_DIFF.append(diff)
        idxs.append(len(GLOBAL_CONCEPTS)-1)
    COURSE_CONCEPT_INDICES[cname]=idxs
N_GLOBAL=len(GLOBAL_CONCEPTS)
print(f'Global pool: {N_GLOBAL} concepts across {len(COURSES)} courses')
for cn,ci in COURSE_CONCEPT_INDICES.items():
    print(f'  {cn:<25}: {len(ci)} concepts')

Global pool: 89 concepts across 5 courses
  algorithms               : 18 concepts
  operating_systems        : 20 concepts
  computer_networks        : 17 concepts
  computer_architecture    : 16 concepts
  software_engineering     : 18 concepts


## Cell 3: Config

In [32]:
class Config:
    N_LEVELS=3; LEVEL_NAMES=['Easy','Medium','Hard']
    GAMMA_LEVEL=np.array([0.0330,0.1494,0.8884])
    RHO_LEVEL  =np.array([0.1444,0.3433,0.2331])
    BETA_LEVEL =np.array([1.4333,1.0389,0.4271])
    LLM_BETA_SCALE=-0.4; LLM_BETA_MID=3.0
    PFA_TOP_K=5; PFA_ALPHA=0.04; SIM_THRESHOLD=0.45
    MAX_STEPS=60; CONCEPT_CAP=20   # MAX_STEPS//3
    # Lower mastery threshold: 0.65 instead of 0.68
    # Starting P(Hard) ≈ 0.605 — only needs 2-3 correct Hard answers
    # Raise SESSION_DONE_BONUS to make goal completion highly valuable
    MASTERY_THRESHOLD=0.65; GOAL_FRACTION=0.30; SESSION_DONE_BONUS=10.0
    EMBED_MODEL='BAAI/bge-base-en-v1.5'; EMBED_DIM=768
    STATE_DIM=14; SCORER_IN=17; HIDDEN_DIM=64
    ITEM_DIFFICULTY=np.array([0.0,1.0,2.2])
    SLIP=0.08; GUESS=0.15; LEARN_RATE=0.10; FORGET_RATE=0.003; TRANSFER_ALPHA=0.25
    # Reward rebalancing:
    # W_BONUS raised to 8.0 — mastery must dominate over step rewards
    # W_CORRECT lowered to 0.3 — reduce safe easy-answer exploitation
    # W_PROGRESS raised to 3.0 — stronger signal for actual PFA improvement
    # W_FIT raised to 0.5 — stronger penalty for Easy on capable students
    # W_DIV raised to 0.4 — stronger diversity to prevent concept fixation
    W_CORRECT=0.3; W_PROGRESS=3.0; W_BONUS=8.0; W_FIT=0.5; W_DIV=0.4
    LR_ACTOR=5e-5; LR_CRITIC=3e-4; GAMMA=0.99; GAE_LAMBDA=0.95
    # LR_ACTOR lowered further — PPO was overwriting BC warm start too fast
    # ENTROPY_COEF raised — encourage exploration of Hard assignments
    # PPO_EPOCHS reduced to 2 — less aggressive updates per episode
    CLIP_EPS=0.10; ENTROPY_COEF=0.05; VALUE_COEF=0.5; PPO_EPOCHS=2; GRAD_CLIP=0.5
    N_EPISODES=5000; WARMUP_EPISODES=300; LOG_EVERY=200; SAVE_EVERY=1000
    P_ONE_COURSE=0.5
    BC_ACTOR_PATH='/kaggle/input/datasets/rehabbb/warmup-pretrained-model/final_bc_actor_full.pt'
    SAVE_PATH='/kaggle/working/'
    DEVICE=DEVICE

cfg=Config()
train_eps=cfg.N_EPISODES-cfg.WARMUP_EPISODES
total_trans=train_eps*cfg.MAX_STEPS
print(f'Training adequacy:')
print(f'  Transitions    : {total_trans:,}')
print(f'  Actions/session: ~{cfg.CONCEPT_CAP*cfg.N_LEVELS}')
print(f'  Visits/action  : ~{total_trans//(cfg.CONCEPT_CAP*cfg.N_LEVELS)}')
print(f'  GPU time (T4)  : ~{cfg.N_EPISODES*cfg.MAX_STEPS//6000} min')
print(f'  Kaggle OK      : {cfg.N_EPISODES*cfg.MAX_STEPS//6000 < 680}')

Training adequacy:
  Transitions    : 282,000
  Actions/session: ~60
  Visits/action  : ~4700
  GPU time (T4)  : ~50 min
  Kaggle OK      : True


## Cell 4: Embeddings + Global Similarity Graph

In [33]:
print(f'Loading {cfg.EMBED_MODEL}')
embed_model=SentenceTransformer(cfg.EMBED_MODEL)
print(f'Embedding {N_GLOBAL} concepts...')
global_embeddings=embed_model.encode(
    GLOBAL_CONCEPTS,normalize_embeddings=True,
    show_progress_bar=True,batch_size=64
).astype(np.float32)
global_sim_matrix=(global_embeddings@global_embeddings.T)
global_neighbours=[]
for i in range(N_GLOBAL):
    s=global_sim_matrix[i].copy(); s[i]=-1
    mask=s>=cfg.SIM_THRESHOLD
    if mask.sum()==0: top=np.array([np.argmax(s)])
    else: top=np.where(mask)[0][np.argsort(-s[mask])[:cfg.PFA_TOP_K]]
    global_neighbours.append(top)
print(f'Embedding shape : {global_embeddings.shape}')
print(f'Avg neighbours  : {np.mean([len(n) for n in global_neighbours]):.1f} (threshold={cfg.SIM_THRESHOLD})')

print('\nSample cross-course pairs (sim > 0.60):')
shown=0
for i in range(N_GLOBAL):
    for j in global_neighbours[i]:
        ci_c=next(c for c,idxs in COURSE_CONCEPT_INDICES.items() if i in idxs)
        cj_c=next(c for c,idxs in COURSE_CONCEPT_INDICES.items() if j in idxs)
        if ci_c!=cj_c and global_sim_matrix[i,j]>0.60:
            print(f'  [{ci_c[:8]}] {GLOBAL_CONCEPTS[i]}')
            print(f'    ↔ [{cj_c[:8]}] {GLOBAL_CONCEPTS[j]}  sim={global_sim_matrix[i,j]:.3f}')
            shown+=1
            if shown>=5: break
    if shown>=5: break

Loading BAAI/bge-base-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 89 concepts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape : (89, 768)
Avg neighbours  : 5.0 (threshold=0.45)

Sample cross-course pairs (sim > 0.60):
  [algorith] time complexity analysis
    ↔ [computer] sequential circuits  sim=0.722
  [algorith] space complexity analysis
    ↔ [software] system design scalability  sim=0.670
  [algorith] recursion
    ↔ [computer] binary and number systems  sim=0.660
  [algorith] divide and conquer
    ↔ [computer] binary and number systems  sim=0.658
  [algorith] quick sort
    ↔ [operatin] paging  sim=0.657


## Cell 5: PFA Tracker (Global + Session Scope)

- Maintains state for ALL N_GLOBAL concepts
- `start_session(indices)` — sets scope, computes goal
- `update(ci,level,correct)` — updates PFA + propagates globally
- `get_state_features(ci)` — 14-scalar input for actor (session-scoped global features)
- Global propagation flows to all similar concepts regardless of course

In [34]:
def sigmoid(x): return 1./(1.+np.exp(-np.clip(x,-15,15)))

class PFATracker:
    def __init__(self,cfg):
        self.cfg=cfg; N=N_GLOBAL; L=cfg.N_LEVELS
        self.gamma_level=cfg.GAMMA_LEVEL.copy()
        self.rho_level=cfg.RHO_LEVEL.copy()
        self.beta_level=cfg.BETA_LEVEL.copy()
        self.beta_concept=np.array([(d-cfg.LLM_BETA_MID)*cfg.LLM_BETA_SCALE
                                     for d in GLOBAL_LLM_DIFF],dtype=np.float32)
        self.successes=np.zeros((N,L),dtype=np.float32)
        self.failures =np.zeros((N,L),dtype=np.float32)
        self.propagation_bonus=np.zeros((N,L),dtype=np.float32)
        self.session_indices=[]; self.session_mastered_before=set()
        self.session_mastered_now=set(); self.goal_new_mastered=0
        self.action_history={}

    def reset_global(self,prior_history=None):
        N,L=N_GLOBAL,self.cfg.N_LEVELS
        if prior_history is None:
            self.successes=np.zeros((N,L),dtype=np.float32)
            self.failures =np.zeros((N,L),dtype=np.float32)
            self.propagation_bonus=np.zeros((N,L),dtype=np.float32)
        else:
            self.successes=prior_history['successes'].copy()
            self.failures =prior_history['failures'].copy()
            self.propagation_bonus=prior_history['bonuses'].copy()

    def start_session(self,session_indices):
        self.session_indices=list(session_indices)
        self.action_history={}
        self.session_mastered_now=set()
        self.session_mastered_before={
            ci for ci in self.session_indices
            if self.predict(ci,2)>self.cfg.MASTERY_THRESHOLD}
        not_yet=len(self.session_indices)-len(self.session_mastered_before)
        self.goal_new_mastered=max(1,round(not_yet*self.cfg.GOAL_FRACTION))
        return {'n_session_concepts':len(self.session_indices),
                'already_mastered':len(self.session_mastered_before),
                'not_yet_mastered':not_yet,
                'goal_new':self.goal_new_mastered}

    def predict(self,ci,level):
        k=level
        z=(self.beta_concept[ci]+self.beta_level[k]
           +self.gamma_level[k]*np.log1p(self.successes[ci,k])
           -self.rho_level[k]*np.log1p(self.failures[ci,k])
           +self.propagation_bonus[ci,k])
        return float(sigmoid(z))

    def predict_all_global(self):
        z=(self.beta_concept[:,None]+self.beta_level[None,:]
           +self.gamma_level[None,:]*np.log1p(self.successes)
           -self.rho_level[None,:]*np.log1p(self.failures)
           +self.propagation_bonus)
        return sigmoid(z)

    def compute_session_apr(self):
        all_p=self.predict_all_global()
        return float(all_p[self.session_indices].mean())

    def count_newly_mastered(self): return len(self.session_mastered_now)
    def goal_met(self): return self.count_newly_mastered()>=self.goal_new_mastered

    def update(self,ci,level,correct):
        k=level; p_before=self.predict(ci,k)
        if correct: self.successes[ci,k]+=1.
        else:       self.failures[ci,k]+=1.
        p_after=self.predict(ci,k)
        p_hard=self.predict(ci,2)
        was_mastered=(ci in self.session_mastered_before or ci in self.session_mastered_now)
        if p_hard>self.cfg.MASTERY_THRESHOLD and not was_mastered:
            self.session_mastered_now.add(ci)
        delta=np.clip(p_after-p_before,-0.1,0.1)
        if abs(delta) > 1e-6:  # skip if no meaningful change
            # Propagate to ALL concepts with sim above threshold
            # (not just top-K, which missed high-sim cross-course pairs)
            sims = global_sim_matrix[ci]  # (N_GLOBAL,)
            mask = (sims >= self.cfg.SIM_THRESHOLD)  # bool mask
            mask[ci] = False  # exclude self
            for j in np.where(mask)[0]:
                sim = sims[j]
                for lvl in range(k+1):
                    self.propagation_bonus[j,lvl] += (
                        self.cfg.PFA_ALPHA * sim / (lvl+1) * delta
                    )
        return p_before,p_after

    def get_state_features(self,ci):
        all_p=self.predict_all_global()
        sess_p=all_p[self.session_indices]
        sess_s=self.successes[self.session_indices]
        sess_f=self.failures[self.session_indices]
        sess_int=sess_s.sum(axis=1)+sess_f.sum(axis=1)
        mean_p_all=float(sess_p.mean()); std_p_all=float(sess_p.std())
        mean_p_easy=float(sess_p[:,0].mean()); mean_p_med=float(sess_p[:,1].mean())
        mean_p_hard=float(sess_p[:,2].mean())
        frac_unexp=float((sess_int==0).mean())
        log_s_tot=float(np.log1p(sess_s.sum())); log_f_tot=float(np.log1p(sess_f.sum()))
        p_easy=float(all_p[ci,0]); p_med=float(all_p[ci,1]); p_hard=float(all_p[ci,2])
        log_s_c=float(np.log1p(self.successes[ci].sum()))
        log_f_c=float(np.log1p(self.failures[ci].sum()))
        global_int=self.successes.sum(axis=1)+self.failures.sum(axis=1)
        practiced=(global_int>0)
        max_sim=float(global_sim_matrix[ci][practiced].max()) if practiced.any() else 0.
        return np.array([mean_p_all,std_p_all,mean_p_easy,mean_p_med,mean_p_hard,
                         frac_unexp,log_s_tot,log_f_tot,
                         p_easy,p_med,p_hard,log_s_c,log_f_c,max_sim],dtype=np.float32)

    def get_critic_state(self):
        feats=np.stack([self.get_state_features(ci) for ci in self.session_indices])
        return feats.mean(axis=0)

tracker=PFATracker(cfg)
tracker.reset_global()
os_idx=COURSE_CONCEPT_INDICES['operating_systems']
info=tracker.start_session(os_idx[:15])
print('PFA Tracker test (OS session, 15 concepts):')
for k,v in info.items(): print(f'  {k}: {v}')
feats=tracker.get_state_features(os_idx[0])
print(f'Feature shape: {feats.shape}  frac_unexplored={feats[5]:.2f}')

PFA Tracker test (OS session, 15 concepts):
  n_session_concepts: 15
  already_mastered: 2
  not_yet_mastered: 13
  goal_new: 4
Feature shape: (14,)  frac_unexplored=1.00


## Cell 6: Session Pre-Selection

Weakest-first ranking with LLM difficulty tie-break. Unmet goal concepts from prior session always included (priority_indices).

In [35]:
def preselect_session_concepts(tracker,candidate_indices,cap,rng,priority_indices=None):
    candidates=list(candidate_indices)
    priority=list(priority_indices) if priority_indices else []
    if len(candidates)<=cap: return candidates
    scores=np.array([tracker.predict(ci,2) for ci in candidates])
    diff_sc=np.array([GLOBAL_LLM_DIFF[ci] for ci in candidates])
    jitter=rng.uniform(0,1e-4,size=len(candidates))
    sort_key=scores-diff_sc*1e-3+jitter
    ranked=[candidates[i] for i in np.argsort(sort_key)]
    selected=[ci for ci in priority if ci in set(candidates)][:cap]
    selected_set=set(selected); rem=cap-len(selected)
    for ci in ranked:
        if rem<=0: break
        if ci not in selected_set:
            selected.append(ci); selected_set.add(ci); rem-=1
    return selected

def sample_session_courses(rng,cfg):
    cnames=list(COURSE_CONCEPT_INDICES.keys())
    if rng.random()<cfg.P_ONE_COURSE: chosen=[rng.choice(cnames)]
    else: chosen=list(rng.choice(cnames,size=2,replace=False))
    cands=[]
    for cn in chosen: cands.extend(COURSE_CONCEPT_INDICES[cn])
    return cands,chosen

# Test
rng_t=np.random.default_rng(42)
tracker.reset_global()
cands,chosen=sample_session_courses(rng_t,cfg)
sel=preselect_session_concepts(tracker,cands,cfg.CONCEPT_CAP,rng_t)
print(f'Selected courses: {chosen}  |  candidates: {len(cands)}  |  after cap: {len(sel)}')
print('Pre-selected (weakest first):')
for ci in sel[:6]:
    c=next(cn for cn,idxs in COURSE_CONCEPT_INDICES.items() if ci in idxs)
    print(f'  [{c[:8]}] {GLOBAL_CONCEPTS[ci]:<35} P(Hard)={tracker.predict(ci,2):.3f}  llm={GLOBAL_LLM_DIFF[ci]}')

Selected courses: [np.str_('software_engineering'), np.str_('computer_networks')]  |  candidates: 35  |  after cap: 20
Pre-selected (weakest first):
  [software] system design scalability           P(Hard)=0.408  llm=5
  [computer] software defined networking         P(Hard)=0.408  llm=5
  [computer] TCP congestion control              P(Hard)=0.507  llm=4
  [computer] TLS SSL handshake                   P(Hard)=0.507  llm=4
  [software] microservices architecture          P(Hard)=0.507  llm=4
  [computer] routing protocols RIP OSPF          P(Hard)=0.507  llm=4


## Cell 7: RealisticStudent (3-Level IRT)

Operates over all N_GLOBAL concepts. Cross-concept knowledge transfer uses the global similarity graph (cross-course transfer included).

In [36]:
class RealisticStudent:
    """
    3-level IRT student calibrated to match PFA probability scale.

    Key fix: ability is initialized so that P_know values match what
    the PFA tracker expects. Previously ability was ~0-centered while
    PFA BETA_LEVEL=[1.43,1.04,0.43] meant baseline P(correct)~0.8.
    Students were answering wrong constantly → PFA never improved.

    Now: ability[ci, level] initialized near BETA_LEVEL[level] so that
    P_know = sigmoid(ability - item_difficulty) ≈ PFA baseline.
    """
    ITEM_DIFFICULTY = np.array([0.0, 1.0, 2.2])   # IRT difficulty per level

    # Ability anchors aligned with PFA BETA_LEVEL
    # sigmoid(anchor - item_diff) ≈ PFA P(correct|level) at beta_concept=0
    ABILITY_ANCHOR = np.array([1.43, 2.04, 2.62])  # Easy, Med, Hard
    # anchor - item_diff = [1.43, 1.04, 0.42] → sigmoid → [0.81, 0.74, 0.60]
    # which matches BETA_LEVEL sigmoid exactly

    def __init__(self, rng, cfg, archetype='mixed', prior_ability=None):
        self.rng = rng; self.cfg = cfg
        self.slip = cfg.SLIP; self.guess = cfg.GUESS
        self.learn_rate = cfg.LEARN_RATE
        self.forget_rate = cfg.FORGET_RATE
        self.transfer_alpha = cfg.TRANSFER_ALPHA
        self.t = 0; self.last_attempt = defaultdict(int)

        SHIFTS = {
            'beginner':     (-0.8, -0.3),   # P_know Easy ≈ 0.55–0.70
            'intermediate': (-0.2,  0.3),   # P_know Easy ≈ 0.75–0.85
            'advanced':     ( 0.3,  0.8),   # P_know Easy ≈ 0.85–0.92
            'mixed':        (-0.8,  0.8),   # full range
        }
        lo, hi = SHIFTS.get(archetype, (-0.8, 0.8))

        if prior_ability is not None:
            self.ability = prior_ability.copy()
        else:
            # Per-concept shift: how far this student is above/below average
            shift = rng.uniform(lo, hi, size=N_GLOBAL)   # (N_GLOBAL,)
            # Ability = anchor + shift, broadcast over levels
            self.ability = self.ABILITY_ANCHOR[None, :] + shift[:, None]
            # Shape: (N_GLOBAL, 3)
            # advanced student: ability ≈ [1.43+0.6, 2.04+0.6, 2.62+0.6]
            #   P_know(Easy)   = sigmoid(2.03 - 0.0) = 0.88
            #   P_know(Medium) = sigmoid(2.64 - 1.0) = 0.84
            #   P_know(Hard)   = sigmoid(3.22 - 2.2) = 0.74  → easily mastered
            # beginner student: ability ≈ [1.43-0.8, 2.04-0.8, 2.62-0.8]
            #   P_know(Easy)   = sigmoid(0.63 - 0.0) = 0.65
            #   P_know(Medium) = sigmoid(1.24 - 1.0) = 0.56
            #   P_know(Hard)   = sigmoid(1.82 - 2.2) = 0.41  → needs work

    @classmethod
    def from_archetype(cls, rng, cfg, archetype='mixed', prior_ability=None):
        return cls(rng, cfg, archetype=archetype, prior_ability=prior_ability)

    def _apply_forgetting(self, ci, k):
        elapsed = self.t - self.last_attempt[(ci, k)]
        if elapsed > 0:
            self.ability[ci, k] = max(
                self.ability[ci, k] - self.forget_rate * np.sqrt(elapsed), -3.0
            )

    def answer(self, ci, level):
        self.t += 1
        self._apply_forgetting(ci, level)
        logit  = self.ability[ci, level] - self.ITEM_DIFFICULTY[level]
        p_know = float(sigmoid(logit))
        true_p = self.guess + (1.0 - self.slip - self.guess) * p_know
        correct = bool(self.rng.random() < true_p)
        self.last_attempt[(ci, level)] = self.t
        return correct, true_p, p_know

    def learn(self, ci, level, correct):
        if correct:
            self.ability[ci, level] += self.learn_rate
            if level > 0:
                self.ability[ci, level-1] += self.learn_rate * 0.3
        else:
            self.ability[ci, level] = max(
                self.ability[ci, level] - self.learn_rate * 0.3, -3.0
            )
        for j in global_neighbours[ci]:
            sim = global_sim_matrix[ci, j]
            if sim < 0.5: continue
            self._apply_forgetting(j, level)
            if correct:
                self.ability[j, level] += self.transfer_alpha * sim * self.learn_rate


# ── Sanity check: verify P_know ranges match PFA expectations ─────────────
rng_test = np.random.default_rng(0)
print('RealisticStudent calibration check:')
print(f'  PFA baseline P(correct): Easy={sigmoid(1.4333):.3f}  '
      f'Med={sigmoid(1.0389):.3f}  Hard={sigmoid(0.4271):.3f}')
print()
for arch in ['beginner','intermediate','advanced']:
    stu = RealisticStudent.from_archetype(rng_test, cfg, archetype=arch)
    # Sample P_know for a random concept at each level
    ci = 0
    pk = [float(sigmoid(stu.ability[ci,k] - stu.ITEM_DIFFICULTY[k])) for k in range(3)]
    print(f'  {arch:<14}: P_know Easy={pk[0]:.3f}  Med={pk[1]:.3f}  Hard={pk[2]:.3f}')
print()
print('Expected:')
print('  beginner    : Easy~0.60-0.70  Med~0.50-0.65  Hard~0.35-0.50')
print('  intermediate: Easy~0.75-0.85  Med~0.65-0.78  Hard~0.50-0.65')
print('  advanced    : Easy~0.85-0.92  Med~0.78-0.88  Hard~0.65-0.80')


RealisticStudent calibration check:
  PFA baseline P(correct): Easy=0.807  Med=0.739  Hard=0.605

  beginner      : P_know Easy=0.721  Med=0.636  Hard=0.485
  intermediate  : P_know Easy=0.831  Med=0.769  Hard=0.642
  advanced      : P_know Easy=0.900  Med=0.858  Hard=0.765

Expected:
  beginner    : Easy~0.60-0.70  Med~0.50-0.65  Hard~0.35-0.50
  intermediate: Easy~0.75-0.85  Med~0.65-0.78  Hard~0.50-0.65
  advanced    : Easy~0.85-0.92  Med~0.78-0.88  Hard~0.65-0.80


## Cell 8: Actor + Critic

In [37]:
class EPPOActor(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.scorer=nn.Sequential(
            nn.Linear(cfg.SCORER_IN,cfg.HIDDEN_DIM),nn.ReLU(),
            nn.Linear(cfg.HIDDEN_DIM,cfg.HIDDEN_DIM//2),nn.ReLU(),
            nn.Linear(cfg.HIDDEN_DIM//2,1))
    def get_policy(self,tracker,session_indices,cfg):
        N_s=len(session_indices); L=cfg.N_LEVELS
        feats=np.stack([tracker.get_state_features(ci) for ci in session_indices])
        ft=torch.tensor(feats,dtype=torch.float32).to(DEVICE)
        do=torch.eye(L,device=DEVICE)
        fe=ft.repeat_interleave(L,dim=0); de=do.repeat(N_s,1)
        x=torch.cat([fe,de],dim=1)
        logits=self.scorer(x).squeeze(-1)
        return Categorical(logits=logits),logits

class EPPOCritic(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(cfg.STATE_DIM,cfg.HIDDEN_DIM),nn.ReLU(),
            nn.Linear(cfg.HIDDEN_DIM,cfg.HIDDEN_DIM//2),nn.ReLU(),
            nn.Linear(cfg.HIDDEN_DIM//2,1))
    def forward(self,x): return self.net(x).squeeze(-1)

actor=EPPOActor(cfg).to(DEVICE)
critic=EPPOCritic(cfg).to(DEVICE)
print(f'Actor params : {sum(p.numel() for p in actor.parameters()):,}')
print(f'Critic params: {sum(p.numel() for p in critic.parameters()):,}')

print(f'\nLoading BC weights from: {cfg.BC_ACTOR_PATH}')
if os.path.exists(cfg.BC_ACTOR_PATH):
    actor.load_state_dict(torch.load(cfg.BC_ACTOR_PATH,map_location=DEVICE))
    print('BC weights loaded.')
else:
    print('WARNING: BC weights not found. Training from random init.')
    print('  Upload bc_actor_full.pt as a Kaggle dataset input.')

# Sanity check
tracker.reset_global(); os_sess=COURSE_CONCEPT_INDICES['operating_systems'][:15]
tracker.start_session(os_sess)
actor.eval()
with torch.no_grad(): _,logits=actor.get_policy(tracker,os_sess,cfg)
print(f'Logit std: {logits.std().item():.4f}  (>0.1 expected if BC loaded)')

Actor params : 3,265
Critic params: 3,073

Loading BC weights from: /kaggle/input/datasets/rehabbb/warmup-pretrained-model/final_bc_actor_full.pt
BC weights loaded.
Logit std: 0.5698  (>0.1 expected if BC loaded)


## Cell 9: Reward Function

All 5 components are LOCAL to the assigned concept — avoids 1/N dilution.

In [38]:
def compute_reward(ci, level, correct, p_before, p_after, action_history, cfg):
    """
    Rebalanced reward function.

    Changes from v1:
    - W_CORRECT lowered: reduces safe easy-answer exploitation
    - W_PROGRESS raised: stronger gradient toward actual PFA movement
    - W_BONUS raised to 8.0: mastery must dominate over step rewards
    - W_FIT raised: stronger push toward Hard for capable students
    - Hard assignment bonus: extra +0.5 when Hard is assigned correctly
      This specifically encourages the agent to attempt Hard assignments
      which are the only path to crossing the mastery threshold.
    """
    # 1. Correctness — lowered to reduce exploitation
    r_correct = cfg.W_CORRECT * float(correct)

    # 2. Local PFA progress — raised
    r_progress = cfg.W_PROGRESS * (p_after - p_before)

    # 3. Hard assignment bonus — new
    # Correct answer at Hard difficulty gets extra reward.
    # This directly incentivises the agent to attempt Hard when appropriate
    # rather than defaulting to safe Easy assignments.
    r_hard_bonus = 0.5 if (level == 2 and correct) else 0.0

    # 4. Difficulty fit
    ideal_level = (1.0 - p_before) * (cfg.N_LEVELS - 1)
    diff_gap    = abs(ideal_level - float(level)) / (cfg.N_LEVELS - 1)
    r_fit       = -cfg.W_FIT * diff_gap

    # 5. Diversity — raised
    key  = (ci, level)
    reps = action_history.get(key, 0)
    r_div = -cfg.W_DIV * min(reps, 4) * 0.25
    action_history[key] = reps + 1

    return r_correct + r_progress + r_hard_bonus + r_fit + r_div


print('Reward function v2 ready.')
print('  W_CORRECT=0.3  W_PROGRESS=3.0  W_BONUS=8.0  W_FIT=0.5  W_DIV=0.4')
print('  Hard correct bonus: +0.5')
print('  Mastery threshold : 0.65  (was 0.68)')
print('  Session done bonus: 10.0  (was 5.0)')


Reward function v2 ready.
  W_CORRECT=0.3  W_PROGRESS=3.0  W_BONUS=8.0  W_FIT=0.5  W_DIV=0.4
  Hard correct bonus: +0.5
  Mastery threshold : 0.65  (was 0.68)
  Session done bonus: 10.0  (was 5.0)


## Cell 10: Rollout Buffer + GAE + PPO Update

In [39]:
class RolloutBuffer:
    def __init__(self): self.clear()
    def clear(self):
        self.critic_states      = []   # (14,) mean-pooled — for critic
        self.actions            = []   # local flat action index
        self.log_probs          = []   # log pi(a|s) — OLD policy, for ratio
        self.rewards            = []
        self.values             = []
        self.dones              = []
        # Store inputs needed to RECOMPUTE logits inside PPO with grad
        # We cannot reuse stored logits — they are detached (no grad_fn)
        self.step_feats_list    = []   # per step: (N_sess, 14) numpy
        self.session_idx_list   = []   # per step: list of global concept indices

    def compute_gae(self, last_value, cfg):
        T=len(self.rewards)
        adv=np.zeros(T,dtype=np.float32); ret=np.zeros(T,dtype=np.float32)
        gae=0.; nv=last_value
        for t in reversed(range(T)):
            mask=1.-float(self.dones[t])
            delta=self.rewards[t]+cfg.GAMMA*nv*mask-self.values[t]
            gae=delta+cfg.GAMMA*cfg.GAE_LAMBDA*mask*gae
            adv[t]=gae; ret[t]=gae+self.values[t]; nv=self.values[t]
        adv=(adv-adv.mean())/(adv.std()+1e-8)
        return adv,ret


def ppo_update(actor, critic, buffer, opt_actor, opt_critic, cfg):
    """
    PPO update — recomputes logits from stored inputs inside gradient context.

    Why we cannot use stored logits:
      During rollout, logits are computed under torch.no_grad() and then
      .detach().cpu() — they have no grad_fn. Calling .backward() on a loss
      built from them raises: 'element 0 of tensors does not require grad'.

    Fix:
      Store (step_feats, session_indices) per step in the buffer.
      Inside PPO, rerun actor.scorer() with grad enabled to get fresh logits.
      Use old_log_probs (stored as scalars) only for the ratio computation.
    """
    # GAE
    with torch.no_grad():
        lv = critic(
            torch.tensor(buffer.critic_states[-1], dtype=torch.float32).to(DEVICE)
        ).item()
    adv_arr, ret_arr = buffer.compute_gae(lv, cfg)

    # Convert fixed tensors (no grad needed)
    cs_t   = torch.tensor(np.stack(buffer.critic_states), dtype=torch.float32).to(DEVICE)
    act_t  = torch.tensor(buffer.actions,   dtype=torch.long   ).to(DEVICE)
    olp_t  = torch.tensor(buffer.log_probs, dtype=torch.float32).to(DEVICE)
    adv_t  = torch.tensor(adv_arr, dtype=torch.float32).to(DEVICE)
    ret_t  = torch.tensor(ret_arr, dtype=torch.float32).to(DEVICE)

    diff_ohs = torch.eye(cfg.N_LEVELS, device=DEVICE)  # (L, L)

    tal = tcl = tent = 0.
    T = len(buffer.actions)

    for _ in range(cfg.PPO_EPOCHS):
        # ── Recompute logits WITH gradients ──────────────────────────────
        # Process each step individually — session scope may vary between eps
        # but within one episode it's fixed, so we batch by episode.
        # For simplicity (and because T<=60) we loop over steps.
        # Session scope is the same for all steps in this episode buffer
        # (one episode = one session = fixed scope)
        # So we can batch all steps at once.
        session_indices = buffer.session_idx_list[0]   # same for all steps
        N_sess = len(session_indices)
        L = cfg.N_LEVELS

        # Stack all step features: (T, N_sess, 14)
        all_feats_np = np.stack(buffer.step_feats_list, axis=0)  # (T, N_sess, 14)
        feats_t = torch.tensor(all_feats_np, dtype=torch.float32).to(DEVICE)  # (T, N_sess, 14)

        # Expand for all difficulty levels: (T, N_sess*L, 14)
        feats_exp = feats_t.repeat_interleave(L, dim=1)          # (T, N_sess*L, 14)

        # Difficulty one-hots: (N_sess*L, L) → (T, N_sess*L, L)
        diff_exp = diff_ohs.repeat(N_sess, 1).unsqueeze(0).expand(T, -1, -1)  # (T, N_sess*L, L)

        # Concat: (T, N_sess*L, 17)
        x = torch.cat([feats_exp, diff_exp], dim=-1)

        # Single batched scorer call — gradients flow through here
        logits = actor.scorer(x).squeeze(-1)  # (T, N_sess*L)  ← HAS grad_fn

        # Log probs and entropy
        from torch.distributions import Categorical
        dist    = Categorical(logits=logits)
        new_lp  = dist.log_prob(act_t)       # (T,)
        entropy = dist.entropy().mean()

        # PPO clipped objective
        ratio = torch.exp(new_lp - olp_t)
        s1    = ratio * adv_t
        s2    = torch.clamp(ratio, 1 - cfg.CLIP_EPS, 1 + cfg.CLIP_EPS) * adv_t
        al    = -torch.min(s1, s2).mean() - cfg.ENTROPY_COEF * entropy

        # Critic loss
        cl = nn.MSELoss()(critic(cs_t), ret_t)

        # Actor update (al has grad_fn through logits → actor.scorer)
        opt_actor.zero_grad()
        al.backward()
        nn.utils.clip_grad_norm_(actor.parameters(), cfg.GRAD_CLIP)
        opt_actor.step()

        # Critic update
        opt_critic.zero_grad()
        cl.backward()
        nn.utils.clip_grad_norm_(critic.parameters(), cfg.GRAD_CLIP)
        opt_critic.step()

        tal  += al.item()
        tcl  += cl.item()
        tent += entropy.item()

    K = cfg.PPO_EPOCHS
    return tal/K, tcl/K, tent/K


print('RolloutBuffer and PPO update fixed (recomputes logits with grad).')


RolloutBuffer and PPO update fixed (recomputes logits with grad).


## Cell 11: Prior History Simulator + Episode Runner

In [40]:
def simulate_prior_history(rng,cfg,n_prior_sessions=0):
    if n_prior_sessions==0: return None
    N=N_GLOBAL; L=cfg.N_LEVELS
    n_prac=rng.integers(5,min(30,N))
    practiced=rng.choice(N,size=n_prac,replace=False)
    s=np.zeros((N,L),dtype=np.float32); f=np.zeros((N,L),dtype=np.float32)
    b=np.zeros((N,L),dtype=np.float32)
    for ci in practiced:
        for k in range(L):
            total=rng.integers(1,3*n_prior_sessions+1)
            pc=rng.uniform(0.4,0.9)
            s[ci,k]=round(total*pc); f[ci,k]=total-s[ci,k]
    return {'successes':s,'failures':f,'bonuses':b}

ARCHETYPES=['beginner','intermediate','advanced','mixed']

def run_episode(actor,critic,tracker,cfg,rng,ep_idx,buffer,is_warmup,priority_indices=None):
    archetype=ARCHETYPES[ep_idx%len(ARCHETYPES)]
    n_prior=int(rng.integers(0,4)) if rng.random()<0.3 else 0
    prior=simulate_prior_history(rng,cfg,n_prior)
    tracker.reset_global(prior_history=prior)
    cands,chosen_courses=sample_session_courses(rng,cfg)
    sess_idx=preselect_session_concepts(tracker,cands,cfg.CONCEPT_CAP,rng,priority_indices)
    sess_info=tracker.start_session(sess_idx)
    N_sess=len(sess_idx)
    student=RealisticStudent.from_archetype(rng,cfg,archetype=archetype)
    action_history={}; ep_reward=0.; mastery_events=[]; buf_ref_levels=[]
    for step in range(cfg.MAX_STEPS):
        cs=tracker.get_critic_state()
        with torch.no_grad():
            value=critic(torch.tensor(cs,dtype=torch.float32).to(DEVICE)).item()
        if is_warmup:
            flat=int(rng.integers(0,N_sess*cfg.N_LEVELS))
            lp=math.log(1./(N_sess*cfg.N_LEVELS))
        else:
            actor.eval()
            with torch.no_grad(): dist,logits=actor.get_policy(tracker,sess_idx,cfg)
            flat=dist.sample().item()
            lp=dist.log_prob(torch.tensor(flat,device=DEVICE)).item()
        local_ci=flat//cfg.N_LEVELS; level=flat%cfg.N_LEVELS
        buf_ref_levels.append(level)
        global_ci=sess_idx[local_ci]
        mb=tracker.count_newly_mastered()
        correct,tp,pk=student.answer(global_ci,level)
        student.learn(global_ci,level,correct)
        p_before,p_after=tracker.update(global_ci,level,int(correct))
        ma=tracker.count_newly_mastered(); newly=(ma>mb)
        reward=compute_reward(global_ci,level,correct,p_before,p_after,action_history,cfg)
        if newly: reward+=cfg.W_BONUS; mastery_events.append(step)
        done=False
        if tracker.goal_met(): reward+=cfg.SESSION_DONE_BONUS; done=True
        elif step==cfg.MAX_STEPS-1: done=True
        # Compute and store per-concept state features for PPO recompute
        step_feats = np.stack(
            [tracker.get_state_features(ci) for ci in sess_idx], axis=0
        )  # (N_sess, 14) — stored as numpy, reused with grad in PPO
        buffer.critic_states.append(cs); buffer.actions.append(flat)
        buffer.log_probs.append(lp); buffer.rewards.append(reward)
        buffer.values.append(value); buffer.dones.append(done)
        buffer.step_feats_list.append(step_feats)
        buffer.session_idx_list.append(sess_idx)
        ep_reward+=reward
        if done: break
    # Difficulty distribution this episode — for diagnosing Easy collapse
    from collections import Counter
    level_counts = Counter(buf_ref_levels)
    return {'reward':ep_reward,'session_apr':tracker.compute_session_apr(),
            'n_mastered':tracker.count_newly_mastered(),'goal_new':sess_info['goal_new'],
            'goal_met':tracker.goal_met(),'steps':step+1,'archetype':archetype,
            'courses':chosen_courses,'n_sess':N_sess,'mastery_events':mastery_events,
            'level_dist':level_counts}

print('Episode runner ready.')

Episode runner ready.


## Cell 12: Training Loop

In [ ]:
def train_eppo(actor,critic,tracker,cfg):
    print('\n'+'='*70)
    print('EPPO FINE-TUNING — Session-Based Architecture')
    print('='*70)
    print(f'  Global concepts : {N_GLOBAL}')
    print(f'  Session cap     : {cfg.CONCEPT_CAP}  Action space/ep: ~{cfg.CONCEPT_CAP*cfg.N_LEVELS}')
    print(f'  Episodes        : {cfg.N_EPISODES}  (warmup: {cfg.WARMUP_EPISODES})')
    print(f'  Steps/episode   : {cfg.MAX_STEPS}')
    print(f'  BC loaded       : {os.path.exists(cfg.BC_ACTOR_PATH)}')
    print()
    opt_actor=optim.Adam(actor.parameters(),lr=cfg.LR_ACTOR)
    opt_critic=optim.Adam(critic.parameters(),lr=cfg.LR_CRITIC)
    buf=RolloutBuffer(); rng=np.random.default_rng(0)
    met={'rewards':[],'session_aprs':[],'n_mastered':[],'goal_rates':[],'steps':[],
         'actor_losses':[],'critic_losses':[],'entropies':[],
         'per_arch':{a:{'rewards':[],'goal_rates':[]} for a in ARCHETYPES}}
    best_reward=-float('inf')
    for ep in range(cfg.N_EPISODES):
        is_warm=ep<cfg.WARMUP_EPISODES; buf.clear()
        info=run_episode(actor,critic,tracker,cfg,rng,ep,buf,is_warm)
        if not is_warm and len(buf.rewards)>1:
            actor.train()
            al,cl,ent=ppo_update(actor,critic,buf,opt_actor,opt_critic,cfg)
            met['actor_losses'].append(al); met['critic_losses'].append(cl)
            met['entropies'].append(ent)
        met['rewards'].append(info['reward']); met['session_aprs'].append(info['session_apr'])
        met['n_mastered'].append(info['n_mastered']); met['goal_rates'].append(float(info['goal_met']))
        met['steps'].append(info['steps'])
        arch=info['archetype']
        met['per_arch'][arch]['rewards'].append(info['reward'])
        met['per_arch'][arch]['goal_rates'].append(float(info['goal_met']))
        if not is_warm and info['reward']>best_reward:
            best_reward=info['reward']
            torch.save(actor.state_dict(),os.path.join(cfg.SAVE_PATH,'eppo_best.pt'))
        if (ep+1)%cfg.SAVE_EVERY==0:
            torch.save(actor.state_dict(),os.path.join(cfg.SAVE_PATH,f'eppo_ep{ep+1}.pt'))
        if (ep+1)%cfg.LOG_EVERY==0:
            sl=slice(-cfg.LOG_EVERY,None); mode='WARM' if is_warm else 'PPO '
            print(f'  [{mode}] ep={ep+1:5d}  '
                  f'R={np.mean(met["rewards"][sl]):+.2f}  '
                  f'APR={np.mean(met["session_aprs"][sl]):.3f}  '
                  f'mastered={np.mean(met["n_mastered"][sl]):.1f}/{round(cfg.CONCEPT_CAP*cfg.GOAL_FRACTION):.0f}  '
                  f'goal%={np.mean(met["goal_rates"][sl])*100:.1f}  '
                  f'steps={np.mean(met["steps"][sl]):.0f}  '
                  f'arch={info["archetype"][:4]}  '
                  f'E={info.get("level_dist",{}).get(0,0):3d} '
                  f'M={info.get("level_dist",{}).get(1,0):3d} '
                  f'H={info.get("level_dist",{}).get(2,0):3d}')
    torch.save(actor.state_dict(),os.path.join(cfg.SAVE_PATH,'eppo_final.pt'))
    print(f'\nDone. Best reward: {best_reward:.3f}')
    return met

metrics=train_eppo(actor,critic,tracker,cfg)


EPPO FINE-TUNING — Session-Based Architecture
  Global concepts : 89
  Session cap     : 20  Action space/ep: ~60
  Episodes        : 5000  (warmup: 300)
  Steps/episode   : 60
  BC loaded       : True

  [WARM] ep=  200  R=+52.93  APR=0.697  mastered=4.8/6  goal%=86.5  steps=38  arch=mixe  E=  3 M=  3 H=  7
  [PPO ] ep=  400  R=+47.99  APR=0.695  mastered=4.5/6  goal%=73.0  steps=45  arch=mixe  E= 12 M= 15 H=  8
  [PPO ] ep=  600  R=+34.63  APR=0.693  mastered=3.4/6  goal%=28.0  steps=56  arch=mixe  E= 32 M= 22 H=  6
  [PPO ] ep=  800  R=+19.26  APR=0.688  mastered=2.0/6  goal%=4.5  steps=60  arch=mixe  E= 34 M= 22 H=  4
  [PPO ] ep= 1000  R=+28.69  APR=0.689  mastered=2.9/6  goal%=16.5  steps=58  arch=mixe  E= 42 M= 10 H=  8
  [PPO ] ep= 1200  R=+36.00  APR=0.691  mastered=3.5/6  goal%=30.5  steps=56  arch=mixe  E= 32 M= 20 H=  8
  [PPO ] ep= 1400  R=+38.06  APR=0.689  mastered=3.6/6  goal%=39.5  steps=55  arch=mixe  E= 43 M=  8 H=  9
  [PPO ] ep= 1600  R=+34.67  APR=0.690  mastered

## Cell 13: Training Curves

In [ ]:
def smooth(x,w=100):
    return np.convolve(x,np.ones(w)/w,mode='valid') if len(x)>w else x

fig,axes=plt.subplots(2,3,figsize=(18,8))
ax=axes[0,0]; ax.plot(smooth(metrics['rewards']),color='#378ADD')
ax.axvline(max(0,cfg.WARMUP_EPISODES-50),color='gray',lw=.8,ls='--',alpha=.5)
ax.set_title('Episode reward (up = better)'); ax.set_xlabel('Episode')
ax=axes[0,1]; ax.plot(smooth(metrics['session_aprs']),color='#1D9E75')
ax.set_title('Session APR (P(correct) within scope)')
ax=axes[0,2]; ax.plot(smooth(metrics['n_mastered']),color='#7F77DD')
tgt=max(1,round(cfg.CONCEPT_CAP*cfg.GOAL_FRACTION))
ax.axhline(tgt,ls='--',color='red',lw=.8,label=f'Goal ({tgt})')
ax.set_title('Concepts newly mastered/session'); ax.legend(fontsize=9)
ax=axes[1,0]; ax.plot(smooth(metrics['goal_rates']),color='#D85A30')
ax.set_title('Goal achievement rate'); ax.set_ylabel('Fraction')
if metrics['actor_losses']:
    ax=axes[1,1]; ax.plot(smooth(metrics['actor_losses']),label='Actor',alpha=.8)
    ax.plot(smooth(metrics['critic_losses']),label='Critic',alpha=.8)
    ax.set_title('PPO losses'); ax.legend(fontsize=9)
if metrics['entropies']:
    ax=axes[1,2]; ax.plot(smooth(metrics['entropies']),color='#854F0B')
    ax.set_title('Policy entropy (should decrease gradually)')
plt.tight_layout(); plt.savefig(os.path.join(cfg.SAVE_PATH,'training_curves.png'),dpi=150)
plt.show()
print('\n=== Per-archetype breakdown (last 500 eps) ===')
for arch in ARCHETYPES:
    ar=metrics['per_arch'][arch]
    if ar['rewards']:
        sl=slice(-500,None)
        print(f'  {arch:<14}: R={np.mean(ar["rewards"][sl]):+.3f}  '
              f'goal%={np.mean(ar["goal_rates"][sl])*100:.1f}%')

## Cell 14: Policy Comparison

In [ ]:
def eval_policy_full(pname,pfn,tracker,cfg,n=200,seed=999):
    rng=np.random.default_rng(seed)
    gains,goals,steps_l,masteries=[],[],[],[]
    for si in range(n):
        arch=ARCHETYPES[si%len(ARCHETYPES)]
        prior=simulate_prior_history(rng,cfg,int(rng.integers(0,3)) if rng.random()<0.3 else 0)
        tracker.reset_global(prior_history=prior)
        cands,_=sample_session_courses(rng,cfg)
        sidx=preselect_session_concepts(tracker,cands,cfg.CONCEPT_CAP,rng)
        tracker.start_session(sidx)
        student=RealisticStudent.from_archetype(rng,cfg,archetype=arch)
        apr_start=tracker.compute_session_apr(); action_history={}
        for step in range(cfg.MAX_STEPS):
            local_ci,level=pfn(tracker,sidx,cfg,rng)
            gci=sidx[local_ci]
            correct,_,_=student.answer(gci,level)
            student.learn(gci,level,correct)
            tracker.update(gci,level,int(correct))
            action_history[(gci,level)]=action_history.get((gci,level),0)+1
            if tracker.goal_met(): steps_l.append(step+1); break
        else: steps_l.append(cfg.MAX_STEPS)
        gains.append(tracker.compute_session_apr()-apr_start)
        goals.append(float(tracker.goal_met()))
        masteries.append(tracker.count_newly_mastered())
    return {'policy':pname,'mean_gain':float(np.mean(gains)),
            'goal_rate':float(np.mean(goals)),'mean_steps':float(np.mean(steps_l)),
            'mean_mastered':float(np.mean(masteries))}

def policy_eppo(t,sidx,cfg,rng):
    actor.eval()
    with torch.no_grad(): dist,_=actor.get_policy(t,sidx,cfg)
    flat=dist.sample().item(); return flat//cfg.N_LEVELS,flat%cfg.N_LEVELS
def policy_random(t,sidx,cfg,rng):
    n=len(sidx); return int(rng.integers(n)),int(rng.integers(cfg.N_LEVELS))
def policy_greedy(t,sidx,cfg,rng):
    pm=[t.predict(ci,1) for ci in sidx]; return int(np.argmin(pm)),1

print('\n=== Policy Comparison (200 sessions each) ===')
print(f'{"Policy":<22} {"ΔAPR":>8} {"Goal%":>7} {"Steps":>7} {"Mastered":>10}')
print('-'*55)
for pn,pf in [('EPPO',policy_eppo),('Random',policy_random),('Greedy-weakest',policy_greedy)]:
    r=eval_policy_full(pn,pf,tracker,cfg,n=200)
    print(f'{pn:<22} {r["mean_gain"]:>+8.4f} {r["goal_rate"]*100:>7.1f}% '
          f'{r["mean_steps"]:>7.1f} {r["mean_mastered"]:>10.1f}')

## Cell 15: Demo Session — Verbose Trace

In [ ]:
def demo_session(course_name,archetype='intermediate',n_prior=0,seed=42):
    rng=np.random.default_rng(seed)
    prior=simulate_prior_history(rng,cfg,n_prior)
    tracker.reset_global(prior_history=prior)
    cands=COURSE_CONCEPT_INDICES[course_name]
    sidx=preselect_session_concepts(tracker,cands,cfg.CONCEPT_CAP,rng)
    info=tracker.start_session(sidx)
    student=RealisticStudent.from_archetype(rng,cfg,archetype=archetype)
    apr_start=tracker.compute_session_apr()
    print(f'\n{"="*85}')
    print(f'SESSION: {course_name.upper()} | {archetype} | {n_prior} prior sessions')
    print(f'  Scope: {info["n_session_concepts"]} concepts  '
          f'(already mastered: {info["already_mastered"]})  '
          f'Goal: master {info["goal_new"]} new  APR start: {apr_start:.4f}')
    print(f'{"="*85}')
    print(f'{"Step":>4} {"Concept":<35} {"Diff":>6} {"P_know":>7} {"OK":>3} {"ΔAPR":>7} {"N/Goal":>7}')
    print('-'*75)
    actor.eval(); action_history={}
    for step in range(cfg.MAX_STEPS):
        mb=tracker.count_newly_mastered(); ab=tracker.compute_session_apr()
        with torch.no_grad(): dist,_=actor.get_policy(tracker,sidx,cfg)
        flat=dist.sample().item(); lci=flat//cfg.N_LEVELS; level=flat%cfg.N_LEVELS
        gci=sidx[lci]; correct,_,pk=student.answer(gci,level)
        student.learn(gci,level,correct); tracker.update(gci,level,int(correct))
        action_history[(gci,level)]=action_history.get((gci,level),0)+1
        aa=tracker.compute_session_apr(); nm=tracker.count_newly_mastered()
        flag=' ★' if nm>mb else ''
        print(f'{step+1:>4} {GLOBAL_CONCEPTS[gci]:<35} {cfg.LEVEL_NAMES[level]:>6} '
              f'{pk:>7.3f} {"✓" if correct else "✗":>3} {aa-ab:>+7.4f} '
              f'{nm:>3}/{info["goal_new"]}{flag}')
        if tracker.goal_met(): print(f'  → GOAL MET at step {step+1}!'); break
    fa=tracker.compute_session_apr()
    print(f'\n  Final APR: {fa:.4f}  ΔAPR={fa-apr_start:+.4f}  '
          f'Mastered: {tracker.count_newly_mastered()}/{info["goal_new"]}  '
          f'Goal: {tracker.goal_met()}')

for course,arch in [('algorithms','beginner'),('operating_systems','intermediate'),
                     ('computer_networks','advanced'),('computer_architecture','mixed'),
                     ('software_engineering','intermediate')]:
    demo_session(course,archetype=arch,n_prior=0,seed=42)

## Cell 16: Cross-Course Knowledge Transfer Verification

In [ ]:
print('\n=== Cross-Course Knowledge Transfer Verification ===')
test_pairs=[
    ('deadlock prevention','operating_systems','process scheduling','operating_systems'),
    ('graph traversal BFS DFS','algorithms','routing fundamentals','computer_networks'),
    ('CPU pipeline stages','computer_architecture','process scheduling','operating_systems'),
]
rng=np.random.default_rng(0)
for cs,crs,ct,crt in test_pairs:
    i=GLOBAL_CONCEPTS.index(cs); j=GLOBAL_CONCEPTS.index(ct)
    sim=global_sim_matrix[i,j]
    tracker.reset_global()
    tracker.start_session(list(range(N_GLOBAL)))
    pb=tracker.predict(j,1)
    for _ in range(5): tracker.update(i,1,correct=1)
    pa=tracker.predict(j,1)
    print(f'\n  [{crs[:6]}] {cs}')
    print(f'    → [{crt[:6]}] {ct}')
    print(f'    sim={sim:.3f}  P(Med) before={pb:.4f}  after={pa:.4f}  '
          f'transfer={"YES" if pa>pb else "NO"}')

## Cell 17: Readiness Checklist

In [ ]:
print('\n'+'='*70)
print('  READINESS CHECKLIST')
print('='*70)
def last200(l): return l[-200:] if len(l)>=200 else l
mr=float(np.mean(last200(metrics['rewards'])))
gr=float(np.mean(last200(metrics['goal_rates'])))*100
mm=float(np.mean(last200(metrics['n_mastered'])))
warm_end=cfg.WARMUP_EPISODES
er=float(np.mean(metrics['rewards'][warm_end:warm_end+200])) if len(metrics['rewards'])>warm_end+200 else mr
checks=[
    ('BC warm start loaded',os.path.exists(cfg.BC_ACTOR_PATH),cfg.BC_ACTOR_PATH),
    ('Reward improved',mr>er,f'{er:+.3f} → {mr:+.3f}'),
    ('Mastered >= 1/session',mm>=1.,f'{mm:.2f}/{round(cfg.CONCEPT_CAP*cfg.GOAL_FRACTION):.0f}'),
    ('Goal rate > 20%',gr>20,f'{gr:.1f}%'),
    ('Session cap enforced',True,f'cap={cfg.CONCEPT_CAP}  {cfg.MAX_STEPS/cfg.CONCEPT_CAP:.1f} steps/concept'),
    ('Global propagation',True,f'{N_GLOBAL} concepts, cross-course active'),
    ('Pre-selection weakest-first',True,'P(Hard) ranked + LLM tie-break'),
    ('Returning student sim',True,'30% of eps have prior history'),
    ('Per-archetype cycling',True,str(ARCHETYPES)),
]
all_pass=True
for label,ok,detail in checks:
    print(f'  [{"PASS" if ok else "FAIL"}] {label:<38} {detail}')
    if not ok: all_pass=False
print()
if not all_pass:
    if not os.path.exists(cfg.BC_ACTOR_PATH):
        print('  BC missing → upload bc_actor_full.pt as Kaggle dataset')
    if mr<=er: print('  No improvement → try N_EPISODES=8000 or LR_ACTOR=3e-4')
    if gr<=20: print('  Low goal rate → reduce GOAL_FRACTION to 0.20 or MAX_STEPS=80')
print()
print('  Production integration:')
print('    1. Replace COURSES with LLM extractor output from uploaded PDFs')
print('    2. preselect_session_concepts() on session start with stored PFA state')
print('    3. Write tracker.successes/failures/bonuses to DB after each session')
print('    4. Unmet goal concepts → priority_indices for next session')
print('='*70)